## Download PJM RT-HRL LMP Data for Virginia pnodes (Past 2 Years)

Downloads real-time hourly LMP data from the PJM API for the 1152 Virginia pnode IDs identified in notebook 01.

**Approach**: Filter requests directly by `pnode_id` in the API query rather than downloading all nodes.
- Sends requests in batches of pnode IDs per month to stay within URL limits
- Date range: April 2024 – March 2026 (past 2 years)
- Stores results in DuckDB

In [18]:
import requests
import pandas as pd
import duckdb
import time
from pathlib import Path
from datetime import datetime
from dateutil.relativedelta import relativedelta

# Find project root by walking up from cwd until we find the 'data' directory
DATA_DIR = next(
    p / "data" for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "data").exists()
)

API_KEY = "9db34bbf6e804cd181d272bbb8329607"
headers = {"Ocp-Apim-Subscription-Key": API_KEY}
BASE_URL = "https://api.pjm.com/api/v1/rt_hrl_lmps"

In [7]:
# Load the 1152 Virginia pnode IDs
va_pnode_ids = pd.read_csv(DATA_DIR / "processed" / "usa_data" / "va_pnode_ids_final.csv")
pnode_id_set = set(va_pnode_ids["pnode_id"].tolist())
pnode_id_list = va_pnode_ids["pnode_id"].tolist()

print(f"Number of Virginia pnode IDs: {len(pnode_id_list)}")

Number of Virginia pnode IDs: 1139


In [9]:
# Set up DuckDB
db_path = DATA_DIR / "processed" / "usa_data" / "pjm_lmp_va_2yr.duckdb"
con = duckdb.connect(str(db_path))

con.execute("""
    CREATE TABLE IF NOT EXISTS rt_hrl_lmps (
        datetime_beginning_utc VARCHAR,
        datetime_beginning_ept VARCHAR,
        pnode_id BIGINT,
        pnode_name VARCHAR,
        voltage VARCHAR,
        equipment VARCHAR,
        type VARCHAR,
        zone VARCHAR,
        system_energy_price_rt DOUBLE,
        total_lmp_rt DOUBLE,
        congestion_price_rt DOUBLE,
        marginal_loss_price_rt DOUBLE,
        row_is_current BOOLEAN,
        version_nbr INTEGER
    )
""")

# Track which months are already done to allow resuming
try:
    done_months = con.execute("""
        SELECT DISTINCT substr(datetime_beginning_ept, 1, 7) AS month
        FROM rt_hrl_lmps
    """).fetchdf()["month"].tolist()
    # Reformat to yyyy-mm
    done_months_set = set(
        f"{m.split('/')[2]}-{m.split('/')[0].zfill(2)}"
        for m in done_months if m
    )
except Exception:
    done_months_set = set()

print(f"Already completed months: {sorted(done_months_set)}")

Already completed months: []


In [ ]:
# Define API fetch and monthly download functions with pagination, rate limiting, and retry logic
def fetch_for_pnode_batch(pnode_ids, date_str, max_retries=5):
    """
    Fetch all rows for a list of pnode_ids for a given date range string.
    Paginates until all rows are retrieved. Retries on 429 with backoff.
    Returns a DataFrame filtered to only the requested pnode_ids.
    """
    all_rows = []
    start_row = 1

    # PJM API accepts multiple pnode_ids as semicolon-separated values
    pnode_id_str = ";".join(str(pid) for pid in pnode_ids)

    while True:
        params = {
            "rowCount": 50000,
            "startRow": start_row,
            "datetime_beginning_ept": date_str,
            "row_is_current": 1,
            "pnode_id": pnode_id_str,
        }

        for attempt in range(max_retries):
            r = requests.get(BASE_URL, headers=headers, params=params)
            if r.status_code == 200:
                break
            elif r.status_code == 429:
                wait = 10 * (2 ** attempt)  # 10s, 20s, 40s, 80s, 160s
                print(f"    429 rate limit — waiting {wait}s (attempt {attempt+1}/{max_retries})", flush=True)
                time.sleep(wait)
            else:
                print(f"    API error {r.status_code}: {r.text[:200]}")
                return pd.DataFrame(all_rows) if all_rows else pd.DataFrame()
        else:
            print(f"    Max retries exceeded, skipping batch")
            break

        items = r.json().get("items", [])
        all_rows.extend(items)

        if len(items) < 50000:
            break
        start_row += 50000
        time.sleep(10)

    if not all_rows:
        return pd.DataFrame()

    df = pd.DataFrame(all_rows)
    # Safety filter: keep only our pnode_ids
    if "pnode_id" in df.columns:
        df = df[df["pnode_id"].isin(pnode_id_set)]
    return df


def download_month_filtered(start_date, pnode_ids, batch_size=50):
    """
    Download all data for a month, split into batches of pnode_ids.
    Returns combined DataFrame for the month.
    """
    end_date = start_date + relativedelta(months=1) - relativedelta(days=1)
    date_str = (
        f"{start_date.month}-{start_date.day}-{start_date.year} 00:00 to "
        f"{end_date.month}-{end_date.day}-{end_date.year} 23:00"
    )

    month_dfs = []
    batches = [pnode_ids[i:i + batch_size] for i in range(0, len(pnode_ids), batch_size)]

    for batch_idx, batch in enumerate(batches):
        df = fetch_for_pnode_batch(batch, date_str)
        if not df.empty:
            month_dfs.append(df)
        if (batch_idx + 1) % 5 == 0:
            print(f"    Batch {batch_idx + 1}/{len(batches)} done", flush=True)
        time.sleep(10)  # 6 connections/min limit = 1 request per 10s

    if not month_dfs:
        return pd.DataFrame()
    return pd.concat(month_dfs, ignore_index=True).drop_duplicates()

In [36]:
# Main download loop: May 2024 – March 2026
# Starting May 2024 to stay safely within the standard data window.
# The rt_hrl_lmps archive cutoff is rolling current_date - 731 days (~2 years).
# As of April 2026, data before ~April 5 2024 is "historic" and cannot be filtered
# by pnode_id. Requests spanning the cutoff boundary return an error.
start_date = datetime(2024, 5, 1)
end_date = datetime(2026, 3, 1)

total_rows = 0
current = start_date

while current <= end_date:
    month_key = current.strftime("%Y-%m")

    if month_key in done_months_set:
        print(f"Skipping {month_key} (already downloaded)")
        current += relativedelta(months=1)
        continue

    print(f"Downloading {month_key}...", flush=True)
    t0 = time.time()

    df = download_month_filtered(current, pnode_id_list, batch_size=50)

    if not df.empty:
        con.execute("INSERT INTO rt_hrl_lmps SELECT * FROM df")
        total_rows += len(df)

    elapsed = time.time() - t0
    print(f"  {month_key}: {len(df):,} rows in {elapsed:.0f}s — cumulative: {total_rows:,}", flush=True)
    current += relativedelta(months=1)

con.close()
print(f"\nDone. Total rows stored: {total_rows:,}")

    Batch 5/23 done
    Batch 10/23 done
    Batch 15/23 done
    Batch 20/23 done
  2024-05: 805,752 rows in 322s — cumulative: 805,752
    Batch 5/23 done
    Batch 10/23 done
    Batch 15/23 done
    Batch 20/23 done
  2024-06: 781,584 rows in 327s — cumulative: 1,587,336
    Batch 5/23 done
    Batch 10/23 done
    Batch 15/23 done
    Batch 20/23 done
  2024-07: 808,728 rows in 327s — cumulative: 2,396,064
    Batch 5/23 done
    Batch 10/23 done
    Batch 15/23 done
    Batch 20/23 done
  2024-08: 808,728 rows in 326s — cumulative: 3,204,792
    Batch 5/23 done
    Batch 10/23 done
    Batch 15/23 done
    Batch 20/23 done
  2024-09: 784,512 rows in 316s — cumulative: 3,989,304
    Batch 5/23 done
    Batch 10/23 done
    Batch 15/23 done
    Batch 20/23 done
  2024-10: 813,192 rows in 332s — cumulative: 4,802,496
    Batch 5/23 done
    Batch 10/23 done
    Batch 15/23 done
    Batch 20/23 done
  2024-11: 788,053 rows in 326s — cumulative: 5,590,549
    Batch 5/23 done
    Batch

In [24]:
#Row count check
con = duckdb.connect(str(db_path))                                                                                                                                                         
con.execute("""                                                                                                                                                                            
    SELECT substr(datetime_beginning_ept, 1, 7) AS month, COUNT(*) as rows                                                                                                                 
    FROM rt_hrl_lmps                                                                                                                                                                       
    GROUP BY month                                                                                                                                                                         
    ORDER BY month                                                                                                                                                                         
""").fetchdf()

,month,rows
0,2024-05,805752
1,2024-06,781584
2,2024-07,808728
3,2024-08,808728
4,2024-09,784512
5,2024-10,813192
6,2024-11,788053
7,2024-12,814200
8,2025-01,814680
9,2025-02,735840


In [ ]:
#Summary checks
con = duckdb.connect(str(db_path))                                                                                                                                                         
print("Total rows:", con.execute("SELECT COUNT(*) FROM rt_hrl_lmps").fetchone()[0])
print("\nDate range:")                                                                                                                                                                     
print(con.execute("SELECT MIN(datetime_beginning_ept), MAX(datetime_beginning_ept) FROM rt_hrl_lmps").fetchone())
print("\nUnique pnode_ids:", con.execute("SELECT COUNT(DISTINCT pnode_id) FROM rt_hrl_lmps").fetchone()[0])                                                                                
print("\nType breakdown:")                                                                                                                                                                 
print(con.execute("SELECT type, COUNT(*) as n FROM rt_hrl_lmps GROUP BY type ORDER BY n DESC").fetchdf())
con.close()   

Total rows: 18466383

Date range:
('2024-05-01T00:00:00', '2026-03-31T23:00:00')

Unique pnode_ids: 1139

Type breakdown:
   type         n
0  LOAD  14517377
1   GEN   3344206
2   EHV    604800


In [26]:
#Check for duplicates
con = duckdb.connect(str(db_path))
con.execute("""
    SELECT COUNT(*) as duplicate_pairs      
    FROM (                                  
        SELECT datetime_beginning_utc, pnode_id
        FROM rt_hrl_lmps                                                                                                                                                                   
        GROUP BY datetime_beginning_utc, pnode_id                                                                                                                                          
        HAVING COUNT(*) > 1                                                                                                                                                                
    )                                                                                                                                                                                      
""").fetchone() 

(0,)

In [27]:
#Backup this raw data
con = duckdb.connect(str(db_path))
con.execute(f"""
    COPY rt_hrl_lmps TO '/Users/elenamurray/Documents/Documents/Repositories/MDS_THESIS/Master-Thesis-2026/data/raw/pjm_lmp_data/pjm_lmp_2yr_raw.parquet' (FORMAT PARQUET)
""")
print("Backup saved")
con.close()

Backup saved
